In [1]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
cwd = Path.cwd()
project_root = cwd.parent
if project_root not in sys.path:
    sys.path.insert(0, str(project_root))
    print("Done!")

Done!


In [3]:
from credit_risk.dataset import load_splits
from credit_risk.config import PROCESSED_DATA_DIR

2026-06-02 08:46:23.820 | INFO     | credit_risk.config:<module>:11 - PROJ_ROOT path is: /Users/ak007/SML/Credit-Risk-Default-Prediction-System


In [4]:
train, val, test, _ = load_splits(PROCESSED_DATA_DIR / "after_eda")

2026-06-02 08:46:35.585 | INFO     | credit_risk.dataset:load_splits:260 - Checking if the files exists...
2026-06-02 08:46:35.593 | INFO     | credit_risk.dataset:load_splits:262 - Loading the Cached files...
2026-06-02 08:46:35.851 | INFO     | credit_risk.dataset:load_splits:270 - Loaded sucessfully all the splits and the metadata, Train_df shape: (466071, 110), val_df shape: (420204, 110), test_df shape: (431712, 110)


In [5]:
check = train.isna().all()

In [6]:
check[check]

annual_inc_joint                       True
dti_joint                              True
verification_status_joint              True
open_acc_6m                            True
open_act_il                            True
open_il_12m                            True
open_il_24m                            True
mths_since_rcnt_il                     True
total_bal_il                           True
il_util                                True
open_rv_12m                            True
open_rv_24m                            True
max_bal_bc                             True
all_util                               True
inq_fi                                 True
total_cu_tl                            True
inq_last_12m                           True
revol_bal_joint                        True
sec_app_fico_range_low                 True
sec_app_fico_range_high                True
sec_app_earliest_cr_line               True
sec_app_inq_last_6mths                 True
sec_app_mort_acc                

In [15]:
for name, df in [("train", train), ("val", val), ("test", test)]:
    fully_null = df.columns[df.isna().all()].tolist()
    print(f"{name}: {len(fully_null)} fully-null columns")
    print(f"  {fully_null}")

train: 30 fully-null columns
  ['annual_inc_joint', 'dti_joint', 'verification_status_joint', 'open_acc_6m', 'open_act_il', 'open_il_12m', 'open_il_24m', 'mths_since_rcnt_il', 'total_bal_il', 'il_util', 'open_rv_12m', 'open_rv_24m', 'max_bal_bc', 'all_util', 'inq_fi', 'total_cu_tl', 'inq_last_12m', 'revol_bal_joint', 'sec_app_fico_range_low', 'sec_app_fico_range_high', 'sec_app_earliest_cr_line', 'sec_app_inq_last_6mths', 'sec_app_mort_acc', 'sec_app_open_acc', 'sec_app_revol_util', 'sec_app_open_act_il', 'sec_app_num_rev_accts', 'sec_app_chargeoff_within_12_mths', 'sec_app_collections_12_mths_ex_med', 'sec_app_mths_since_last_major_derog']
val: 13 fully-null columns
  ['revol_bal_joint', 'sec_app_fico_range_low', 'sec_app_fico_range_high', 'sec_app_earliest_cr_line', 'sec_app_inq_last_6mths', 'sec_app_mort_acc', 'sec_app_open_acc', 'sec_app_revol_util', 'sec_app_open_act_il', 'sec_app_num_rev_accts', 'sec_app_chargeoff_within_12_mths', 'sec_app_collections_12_mths_ex_med', 'sec_app_mt

In [16]:
joint_loans = train[train['application_type'] == 'Joint App']
print(f"Joint apps in train: {len(joint_loans)}")
print(joint_loans[['revol_bal_joint', 'sec_app_fico_range_high', 'annual_inc_joint']].head())

Joint apps in train: 0
Empty DataFrame
Columns: [revol_bal_joint, sec_app_fico_range_high, annual_inc_joint]
Index: []


In [17]:
for name, df in [("train", train), ("val", val), ("test", test)]:
    print(f"\n{name}:")
    print(df['application_type'].value_counts(dropna=False))


train:
application_type
Individual    466071
Name: count, dtype: int64

val:
application_type
Individual    419694
Joint App        510
Name: count, dtype: int64

test:
application_type
Individual    423032
Joint App       8680
Name: count, dtype: int64


In [9]:
train.select_dtypes(include='str').columns

Index(['term', 'grade', 'sub_grade', 'emp_title', 'emp_length',
       'home_ownership', 'verification_status', 'desc', 'purpose', 'title',
       'zip_code', 'addr_state', 'earliest_cr_line', 'initial_list_status',
       'application_type', 'verification_status_joint',
       'sec_app_earliest_cr_line', 'disbursement_method'],
      dtype='str')

In [32]:
set(train['emp_length'].dropna().unique()).union(set(val['emp_length'].dropna().unique())).union(set(test['emp_length'].dropna().unique()))

{'1 year',
 '10+ years',
 '2 years',
 '3 years',
 '4 years',
 '5 years',
 '6 years',
 '7 years',
 '8 years',
 '9 years',
 '< 1 year'}

In [11]:
train['term'].value_counts()

term
36 months    337996
60 months    128075
Name: count, dtype: int64